In [3]:
import sys
sys.path.append("..")
import pandas as pd
import pdblp
import json
print(pdblp.__version__)

0.1.8+pinnaclefork.1


In [7]:
bcon = pdblp.BCon(timeout=50000, logdir='.', dummy=True)
bcon.start()

# Generate Example API Response

In [8]:
def df_to_nested_dict(df: pd.DataFrame) -> dict:
    """
    Convert a pandas DataFrame with a MultiIndex on columns
    (ticker, field) and a DateTimeIndex to:
      { ticker: { "dates": [...], field1: [...], ... }, ... }
    """
    out = {}
    for ticker in df.columns.levels[0]:
        sub = {"dates": []}
        # initialize field lists
        for fld in df[ticker].columns:
            sub[fld] = []
        # populate
        for date, row in df[ticker].iterrows():
            sub["dates"].append(date)
            for fld in df[ticker].columns:
                val = row[fld]
                sub[fld].append(None if pd.isna(val) else float(val))
        out[ticker] = sub
    return out

In [9]:
## Generate bdh example
bdh_list = [
    (["JCI INDEX", "LQ45 INDEX"], ["PX_LAST", "PX_OPEN", "TURNOVER"]),
    (["BBCA IJ EQUITY", "TLKM IJ EQUITY"], ["NET_INCOME"])
]
START_DATE, END_DATE = "20030101", "20241231"
bdh_examples = []
for tickers, fields in bdh_list:
    df = bcon.bdh(tickers, fields, START_DATE, END_DATE)
    nested = df_to_nested_dict(df)
    bdh_examples.append({
        "args": {
            "tickers": tickers,
            "flds": fields,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "optional_args": {}   # add any extra opts here
        },
        "response": nested
    })

In [ ]:
def df_to_nested_bulkref(df: pd.DataFrame) -> dict:
    """
    Convert the flat bulkref DataFrame into:
      {
        ticker1: {
          field1: [ {name:…, value:…, position:…}, … ],
          field2: [ … ]
        },
        ticker2: { … }
      }
    """
    out = {}
    for ticker in df["ticker"].unique():
        sub = {}
        df_t = df[df["ticker"] == ticker]
        for fld in df_t["field"].unique():
            rows = df_t[df_t["field"] == fld]
            items = []
            for _, r in rows.iterrows():
                items.append({
                    "name":     r["name"],
                    "value":    None if pd.isna(r["value"]) else r["value"],
                    "position": int(r["position"])
                })
            sub[fld] = items
        out[ticker] = sub
    return out

In [ ]:
bulkref_list = [
    (["JCI INDEX", "LQ45 INDEX"], ["INDEX_MEMBERS_SHARES", "INDX_MWEIGHT_PX"]),
    (["BBCA IJ EQUITY", "TLKM IJ EQUITY"], ["EQY_DVD_ADJUST_FACT", "ERN_ANN_DT_AND_PER"])
]
bulkref_examples = []
for tickers, fields in bulkref_list:
    print(f"Fetching bulkref for {tickers} / {fields} …")
    df = bcon.bulkref(tickers=tickers, flds=fields)

    nested = df_to_nested_bulkref(df)
    bulkref_examples.append({
        "args": {
            "tickers": tickers,
            "flds": fields,
            "optional_args": {}
        },
        "response": nested
    })

In [ ]:
def df_to_nested_ref(df: pd.DataFrame) -> dict:
    """
    Convert a flat ref DataFrame into:
      {
        ticker1: { field1: value1, field2: value2, … },
        ticker2: { … }
      }
    """
    out = {}
    for ticker in df["ticker"].unique():
        df_t = df[df["ticker"] == ticker]
        sub = {row["field"]: (None if pd.isna(row["value"]) else row["value"])
               for _, row in df_t.iterrows()}
        out[ticker] = sub
    return out

In [ ]:
ref_list = [
    (["BBCA IJ EQUITY", "TLKM IJ EQUITY"], ["EQY_INIT_PO_DT", "EQY_INIT_PO_SH_PX"]),
    (["BBCA IJ EQUITY", "TLKM IJ EQUITY"], ["ID_ISIN", "PX_LAST"]),
]
ref_examples = []

for tickers, fields in ref_list:
    print(f"Fetching ref for {tickers} / {fields} …")
    df = bcon.ref(tickers=tickers, flds=fields)

    nested = df_to_nested_ref(df)
    ref_examples.append({
        "args": {
            "tickers": tickers,
            "flds": fields,
            "optional_args": {}
        },
        "response": nested
    })

In [ ]:
dummy_examples = {
    "bdh":{
        "params": ["tickers", "flds", "start_date", "end_date", "optional_args"],
        "examples": bdh_examples
    },
    "ref":{
        "params": ["tickers", "flds", "optional_args"],
        "examples": ref_examples
    },
    "bulkref":{
        "params": ["tickers", "flds", "optional_args"],
        "examples": bulkref_examples
    }
}

In [ ]:
with open("dummy_bcon.json", "w") as f:
    json.dump(dummy_examples, f, indent=2, default=str)

# Load Dummy Response

In [11]:
with open("dummy_response.json", "r") as f:
    dummy = json.load(f)

In [12]:
## ChatGPT Generated Parse Function

def parse_bdh_response(resp: dict) -> pd.DataFrame:
    """
    resp: {
      "TICKER1": {
        "dates":   ["2024-01-02", "2024-01-03", …],
        "PX_LAST": [399.17,       394.94,      …],
        "PX_OPEN": [402.39,       396.53,      …]
      },
      "TICKER2": { … }
    }
    returns a DataFrame with
      - DateTimeIndex (dates)
      - MultiIndex columns (ticker, field)
    """
    # build a dict of DataFrames, one per ticker
    dfs = {}
    for ticker, info in resp.items():
        df = pd.DataFrame({
            fld: info[fld]
            for fld in info.keys() if fld != "dates"
        }, index=pd.to_datetime(info["dates"]))
        # assign ticker as top‐level column
        df.columns = pd.MultiIndex.from_product([[ticker], df.columns])
        dfs[ticker] = df

    # concatenate along columns
    full = pd.concat(dfs.values(), axis=1)
    # sort by ticker then field
    full = full.sort_index(axis=1, level=[0,1])
    return full


def parse_ref_response(resp: dict) -> pd.DataFrame:
    """
    resp: {
      "TICKER1": {"FIELD1": val1, "FIELD2": val2, …},
      "TICKER2": { … }
    }
    returns a DataFrame with columns ['ticker','field','value']
    """
    rows = []
    for ticker, fields in resp.items():
        for fld, val in fields.items():
            rows.append({"ticker": ticker, "field": fld, "value": val})
    return pd.DataFrame(rows, columns=["ticker", "field", "value"])


def parse_bulkref_response(resp: dict) -> pd.DataFrame:
    """
    resp: {
      "TICKER1": {
        "FIELD1": [ { "name":…, "value":…, "position":… }, … ],
        "FIELD2": [ … ]
      },
      "TICKER2": { … }
    }
    returns a DataFrame with columns ['ticker','field','name','value','position']
    """
    rows = []
    for ticker, fld_dict in resp.items():
        for fld, items in fld_dict.items():
            for item in items:
                rows.append({
                    "ticker":  ticker,
                    "field":   fld,
                    "name":    item["name"],
                    "value":   item["value"],
                    "position": item["position"]
                })
    return pd.DataFrame(rows, columns=["ticker", "field", "name", "value", "position"])

In [13]:
parse_bdh_response(dummy["bdh"]["examples"][0]["response"])

JCI INDEX                         LQ45 INDEX                       
             PX_LAST   PX_OPEN      TURNOVER    PX_LAST  PX_OPEN      TURNOVER
2003-01-02   409.125   424.945  1.902623e+08     87.836   91.978  1.657212e+08
2003-01-03   407.512   409.125  1.445343e+08     87.508   87.836  1.349981e+08
2003-01-06   398.247   407.512  3.174285e+08     85.285   87.508  2.950308e+08
2003-01-07   394.519   398.247  1.819913e+08     84.292   85.285  1.722716e+08
2003-01-08   389.414   394.519  1.672031e+08     82.698   84.292  1.599545e+08
...              ...       ...           ...        ...      ...           ...
2024-12-20  6983.865  6980.175  9.741628e+09    817.016  818.037  6.132290e+09
2024-12-23  7096.445  7037.530  7.168814e+09    835.752  825.849  3.624156e+09
2024-12-24  7065.746  7115.637  6.302316e+09    830.487  837.975  2.974220e+09
2024-12-27  7036.571  7073.375  5.742560e+09    825.135  830.158  3.003234e+09
2024-12-30  7079.905  7026.777  7.146936e+09    826.647  821.765  4.059669e+09

[5344 rows x 6 columns]

In [14]:
parse_ref_response(dummy["ref"]["examples"][0]["response"])

,ticker,field,value
0,BBCA IJ EQUITY,EQY_INIT_PO_DT,2000-05-31
1,BBCA IJ EQUITY,EQY_INIT_PO_SH_PX,1400.0
2,TLKM IJ EQUITY,EQY_INIT_PO_DT,1995-11-14
3,TLKM IJ EQUITY,EQY_INIT_PO_SH_PX,2050.0


In [16]:
parse_bulkref_response(dummy["bulkref"]["examples"][1]["response"])

,ticker,field,name,value,position
0,BBCA IJ EQUITY,EQY_DVD_ADJUST_FACT,Adjustment Date,2021-10-13,0
1,BBCA IJ EQUITY,EQY_DVD_ADJUST_FACT,Adjustment Factor,5.0,0
2,BBCA IJ EQUITY,EQY_DVD_ADJUST_FACT,Adjustment Factor Operator Type,1.0,0
3,BBCA IJ EQUITY,EQY_DVD_ADJUST_FACT,Adjustment Factor Flag,3.0,0
4,BBCA IJ EQUITY,EQY_DVD_ADJUST_FACT,Adjustment Date,2008-01-28,1
...,...,...,...,...,...
823,TLKM IJ EQUITY,ERN_ANN_DT_AND_PER,Earnings Year and Period,2003:Q2,197
824,TLKM IJ EQUITY,ERN_ANN_DT_AND_PER,Earnings Announcement Date,2003-04-30,198
825,TLKM IJ EQUITY,ERN_ANN_DT_AND_PER,Earnings Year and Period,2003:C1,198
826,TLKM IJ EQUITY,ERN_ANN_DT_AND_PER,Earnings Announcement Date,2003-04-30,199
